# Overfitting How to Detect and Fix It


---

### References
1. Geman et al. (1992) Neural networks and the bias/variance dilemma. https://doi.org/10.1162/neco.1992.4.1.1
2. Hastie, Tibshirani & Friedman (2009) The Elements of Statistical Learning. Springer.
3. Domingos (2000) A unified bias-variance decomposition. ICML.
4. Srivastava et al. (2014) Dropout. JMLR.
5. Bergstra & Bengio (2012) Random search for hyper-parameter optimisation. JMLR.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits, load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    learning_curve, validation_curve, cross_val_score,
    StratifiedKFold, train_test_split
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

os.makedirs('figures_of', exist_ok=True)

# Design system (dark premium theme)
BG      = '#0F1117'
PANEL   = '#1A1D2E'
BORDER  = '#2D3154'
TEXT    = '#E8EAF6'
SUBTEXT = '#8892B0'
GRID    = '#1E2140'

# Accessible colours
C_UNDER  = '#FFB86C'   # amber  underfit
C_GOOD   = '#43D9AD'   # teal   good fit
C_OVER   = '#FF6B6B'   # red    overfit
C_TRAIN  = '#4F9CF9'   # blue   train
C_VAL    = '#BD93F9'   # purple validation
C_GAP    = '#FF6B6B'   # red    gap
C_GREEN  = '#43D9AD'
C_AMBER  = '#FFB86C'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': PANEL,
    'axes.edgecolor': BORDER, 'axes.labelcolor': SUBTEXT,
    'axes.titlecolor': TEXT, 'xtick.color': SUBTEXT,
    'ytick.color': SUBTEXT, 'text.color': TEXT,
    'grid.color': GRID, 'grid.linewidth': 0.6,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.facecolor': PANEL, 'legend.edgecolor': BORDER,
    'legend.labelcolor': TEXT,
})

def style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.grid(True, alpha=0.18)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)

print('Setup complete.')

Setup complete.


## Figure 1 The Three Regimes: Underfit / Good / Overfit

In [ ]:
from numpy.polynomial import polynomial as P

np.random.seed(42)
n = 30
X_raw = np.linspace(0, 1, n)
y_true = np.sin(2 * np.pi * X_raw)
y_noisy = y_true + np.random.normal(0, 0.3, n)
X_test_raw = np.linspace(0, 1, 200)
y_test_true = np.sin(2 * np.pi * X_test_raw)

degrees = [1, 4, 18]
colours = [C_UNDER, C_GOOD, C_OVER]
labels  = ['Underfitting  (degree=1)\nHigh bias misses the pattern',
           'Good fit  (degree=4)\nBias-variance balance',
           'Overfitting  (degree=18)\nMemorises noise fails on new data']
badge_cols = [C_UNDER, C_GOOD, C_OVER]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), gridspec_kw={'wspace': 0.1})
fig.patch.set_facecolor(BG)

for ax, deg, col, lbl, bc in zip(axes, degrees, colours, labels, badge_cols):
    style_ax(ax)
    coeffs = np.polyfit(X_raw, y_noisy, deg)
    poly   = np.poly1d(coeffs)
    y_fit  = poly(X_test_raw)
    train_mse = np.mean((poly(X_raw) - y_noisy)**2)
    test_mse  = np.mean((y_fit - y_test_true)**2)

    ax.plot(X_test_raw, y_test_true, '--', color=SUBTEXT, lw=1.8, alpha=0.6, label='True function')
    ax.scatter(X_raw, y_noisy, color=TEXT, s=35, zorder=4, alpha=0.85,
               edgecolors=BORDER, lw=0.5, label='Train data')
    ax.plot(X_test_raw, np.clip(y_fit, -2.5, 2.5), color=col, lw=2.8,
            label=f'Degree {deg}',
            path_effects=[pe.withStroke(linewidth=5, foreground=col+'44')])

    ax.set_xlim(0, 1); ax.set_ylim(-2.4, 2.4)
    ax.set_xlabel('x', color=SUBTEXT)
    ax.set_ylabel('y', color=SUBTEXT)
    ax.set_title(lbl, fontsize=11, fontweight='bold', color=col, pad=14)

    # MSE badge inside plot
    ax.text(0.03, 0.06,
            f'Train MSE: {train_mse:.3f}\nTest  MSE: {test_mse:.3f}',
            transform=ax.transAxes, fontsize=9.5, color=TEXT,
            verticalalignment='bottom', fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=PANEL,
                      edgecolor=bc, lw=1.5, alpha=0.95))

    ax.legend(fontsize=9, loc='upper right',
              facecolor=PANEL, edgecolor=BORDER)

fig.text(0.5, 1.01, 'The Three Regimes Underfitting, Good Fit, Overfitting',
         ha='center', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.975, 'Overfitting: near-zero train error, high test error',
         ha='center', fontsize=10.5, color=SUBTEXT)

plt.savefig('figures_of/fig1_three_regimes.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show()
print('Figure 1 saved.')

Figure 1 saved.


## Figure 2 Learning Curve + Validation Curve

In [ ]:
X_dig, y_dig = load_digits(return_X_y=True)
clf_rf = RandomForestClassifier(n_estimators=100, random_state=42)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Learning curve
sizes = np.linspace(0.1, 1.0, 10)
tr_sizes, tr_scores, cv_scores = learning_curve(
    clf_rf, X_dig, y_dig,
    train_sizes=sizes, cv=cv5, scoring='accuracy', n_jobs=-1)

tr_mean = tr_scores.mean(axis=1) * 100
tr_std  = tr_scores.std(axis=1)  * 100
cv_mean = cv_scores.mean(axis=1) * 100
cv_std  = cv_scores.std(axis=1)  * 100

# Validation curve max_depth
depths = [1, 2, 3, 5, 8, 12, 18, None]
depth_labels = ['1','2','3','5','8','12','18','∞']
tr_v, cv_v = validation_curve(
    RandomForestClassifier(n_estimators=50, random_state=42),
    X_dig, y_dig,
    param_name='max_depth', param_range=depths,
    cv=cv5, scoring='accuracy', n_jobs=-1)

tr_v_m = tr_v.mean(axis=1) * 100
cv_v_m = cv_v.mean(axis=1) * 100
gap_v  = tr_v_m - cv_v_m

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.8),
                                gridspec_kw={'wspace': 0.32})
fig.patch.set_facecolor(BG)

#Left: learning curve
style_ax(ax1)
ax1.fill_between(tr_sizes, tr_mean-tr_std, tr_mean+tr_std,
                 alpha=0.15, color=C_TRAIN)
ax1.fill_between(tr_sizes, cv_mean-cv_std, cv_mean+cv_std,
                 alpha=0.15, color=C_VAL)
ax1.plot(tr_sizes, tr_mean, '-o', color=C_TRAIN, lw=2.5, ms=6,
         markeredgecolor='white', mew=0.7, label='Train')
ax1.plot(tr_sizes, cv_mean, '-s', color=C_VAL,   lw=2.5, ms=6,
         markeredgecolor='white', mew=0.7, label='CV test')
ax1.fill_between(tr_sizes, cv_mean, tr_mean,
                 alpha=0.18, color=C_GAP, label='Overfit gap')
ax1.set_xlabel('Training set size', color=SUBTEXT, fontsize=11)
ax1.set_ylabel('Accuracy (%)', color=SUBTEXT, fontsize=11)
ax1.set_title('Learning Curve UCI Digits\nGap narrows → overfit, not underfit',
              fontsize=11, fontweight='bold', color=TEXT, pad=10)
ax1.legend(fontsize=10)
ax1.set_ylim(60, 103)

# Right: validation curve
style_ax(ax2)
x_pos = np.arange(len(depths))
ax2.plot(x_pos, tr_v_m, '-o', color=C_TRAIN, lw=2.5, ms=7,
         markeredgecolor='white', mew=0.7, label='Train')
ax2.plot(x_pos, cv_v_m, '-s', color=C_VAL,   lw=2.5, ms=7,
         markeredgecolor='white', mew=0.7, label='CV test')

# Shade gap
ax2.fill_between(x_pos, cv_v_m, tr_v_m,
                 alpha=0.18, color=C_GAP, label='Overfit gap')

# Best depth annotation
best_idx = np.argmax(cv_v_m)
ax2.axvline(best_idx, color=C_GOOD, lw=1.8, linestyle='--', alpha=0.8)
ax2.text(best_idx+0.15, cv_v_m[best_idx]+0.5,
         f'Best=\ndepth {depth_labels[best_idx]}',
         color=C_GOOD, fontsize=9, fontweight='bold')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(depth_labels)
ax2.set_xlabel('max_depth', color=SUBTEXT, fontsize=11)
ax2.set_ylabel('Accuracy (%)', color=SUBTEXT, fontsize=11)
ax2.set_title('Validation Curve max_depth\nReal UCI Digits RF',
              fontsize=11, fontweight='bold', color=TEXT, pad=10)
ax2.legend(fontsize=10)

fig.text(0.5, 1.01,
         'Diagnostic Tools: Learning Curve and Validation Curve',
         ha='center', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.975,
         'Left: does more data help? | Right: is the model too complex?',
         ha='center', fontsize=10.5, color=SUBTEXT)

plt.savefig('figures_of/fig2_learning_validation_curves.png',
            dpi=180, bbox_inches='tight', facecolor=BG)
plt.show()
print('Figure 2 saved.')

Figure 2 saved.


## Figure 3 Five Fixes on Wisconsin Breast Cancer

In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

configs = [
    ('Overfit\n(depth=∞)', DecisionTreeClassifier(random_state=42)),
    ('depth=5\n(reduce complexity)', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('min_leaf=5\n(regularise)',     DecisionTreeClassifier(min_samples_leaf=5, random_state=42)),
    ('n_est=20\n(ensemble)',         RandomForestClassifier(n_estimators=20, random_state=42)),
    ('Best combo\n(RF tuned)',       RandomForestClassifier(n_estimators=100, max_depth=8,
                                                             min_samples_leaf=3, random_state=42)),
]

results = []
for name, clf in configs:
    pipe = Pipeline([('sc', StandardScaler()), ('clf', clf)])
    tr_s  = cross_val_score(pipe, X_bc, y_bc, cv=cv5, scoring='accuracy') * 100
    # Train score: fit on full training fold
    tr_full = []
    for tr_idx, te_idx in cv5.split(X_bc, y_bc):
        pipe.fit(X_bc[tr_idx], y_bc[tr_idx])
        tr_full.append(pipe.score(X_bc[tr_idx], y_bc[tr_idx]) * 100)
    results.append({
        'name':  name,
        'train': np.mean(tr_full),
        'cv':    tr_s.mean(),
        'gap':   np.mean(tr_full) - tr_s.mean()
    })
    print(f"{name.replace(chr(10),' '):25s}  train={np.mean(tr_full):.1f}%  cv={tr_s.mean():.1f}%  gap={np.mean(tr_full)-tr_s.mean():.1f}%")

names = [r['name'] for r in results]
trains= [r['train'] for r in results]
cvs   = [r['cv']    for r in results]
gaps  = [r['gap']   for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.0),
                                gridspec_kw={'wspace': 0.35})
fig.patch.set_facecolor(BG)
x = np.arange(len(names))
w = 0.38

# Left: grouped bars
style_ax(ax1)
b1 = ax1.bar(x - w/2, trains, w, color=C_TRAIN, alpha=0.85,
              label='Train', edgecolor='white', linewidth=0.5)
b2 = ax1.bar(x + w/2, cvs,    w, color=C_VAL,   alpha=0.85,
              label='CV test', edgecolor='white', linewidth=0.5,
              hatch='//')

for bar, val in zip(b1, trains):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.0f}', ha='center', va='bottom', fontsize=9,
             color=TEXT, fontweight='bold')
for bar, val in zip(b2, cvs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.0f}', ha='center', va='bottom', fontsize=9,
             color=TEXT, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(names, fontsize=9)
ax1.set_ylabel('Accuracy (%)', color=SUBTEXT, fontsize=11)
ax1.set_title('Fixes on Real Breast Cancer\nTrain vs CV test accuracy',
              fontsize=11, fontweight='bold', color=TEXT, pad=10)
ax1.legend(fontsize=10)
ax1.set_ylim(84, 104)

# Right: gap bars
style_ax(ax2)
gap_colours = [C_GOOD if g < 2 else (C_AMBER if g < 5 else C_OVER) for g in gaps]
gap_hatches = ['', '//', 'xx', '..', '']
bars = ax2.bar(x, gaps, 0.55, color=gap_colours, alpha=0.85,
               edgecolor='white', linewidth=0.5)
for bar, h in zip(bars, gap_hatches):
    bar.set_hatch(h)

for bar, g in zip(bars, gaps):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f'{g:.1f}%', ha='center', va='bottom', fontsize=10,
             color=TEXT, fontweight='bold')

ax2.axhline(2.0, color=SUBTEXT, linestyle='--', lw=1.5, alpha=0.7,
            label='Target gap <2%')
ax2.set_xticks(x)
ax2.set_xticklabels(names, fontsize=9)
ax2.set_ylabel('Train Test gap (%)', color=SUBTEXT, fontsize=11)
ax2.set_title('Overfitting Gap by Fix\nGreen=healthy, red=overfit',
              fontsize=11, fontweight='bold', color=TEXT, pad=10)
ax2.legend(fontsize=10)

# Colour legend
leg = [mpatches.Patch(color=C_OVER,  label='>5% overfit'),
       mpatches.Patch(color=C_AMBER, label='2–5% borderline'),
       mpatches.Patch(color=C_GOOD,  label='<2% healthy')]
ax2.legend(handles=leg, fontsize=9, loc='upper right')

fig.text(0.5, 1.01, 'Five Regularisation Fixes on Real Wisconsin Breast Cancer',
         ha='center', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.975,
         'Limiting depth and leaves reduces gap without hurting test accuracy',
         ha='center', fontsize=10.5, color=SUBTEXT)

plt.savefig('figures_of/fig3_five_fixes.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show()
print('Figure 3 saved.')

Overfit (depth=∞)          train=100.0%  cv=91.0%  gap=9.0%


depth=5 (reduce complexity)  train=99.2%  cv=92.8%  gap=6.4%
min_leaf=5 (regularise)    train=97.6%  cv=92.8%  gap=4.8%


n_est=20 (ensemble)        train=99.9%  cv=95.8%  gap=4.1%


Best combo (RF tuned)      train=99.1%  cv=95.3%  gap=3.8%


Figure 3 saved.


## Figure 4 Bias-Variance Decomposition Visualised

In [ ]:
# Simulate bias-variance tradeoff across model complexity
complexity = np.linspace(1, 10, 100)
bias2    = 4.0 / complexity**1.4
variance = 0.05 * complexity**1.8
irred    = np.full_like(complexity, 0.1)
total    = bias2 + variance + irred

train_err = variance * 0.15 + irred
test_err  = total

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5),
                          gridspec_kw={'wspace': 0.32})
fig.patch.set_facecolor(BG)

# Left: decomposition
ax = axes[0]
style_ax(ax)
ax.stackplot(complexity, bias2, variance, irred,
             labels=['Bias²', 'Variance', 'Irreducible noise'],
             colors=[C_OVER, C_TRAIN, SUBTEXT], alpha=0.55)
ax.plot(complexity, total, color='white', lw=2.5, label='Total error')
best = np.argmin(total)
ax.axvline(complexity[best], color=C_GOOD, lw=2, linestyle='--')
ax.text(complexity[best]+0.2, total[best]+0.05,
        'Sweet spot', color=C_GOOD, fontsize=10, fontweight='bold')
ax.set_xlabel('Model complexity', color=SUBTEXT, fontsize=11)
ax.set_ylabel('Expected error', color=SUBTEXT, fontsize=11)
ax.set_title('Bias-Variance Decomposition\nExpected error = Bias² + Variance + Noise',
             fontsize=11, fontweight='bold', color=TEXT, pad=10)
ax.legend(fontsize=10)

#  Right: train vs test error curves
ax2 = axes[1]
style_ax(ax2)
ax2.plot(complexity, train_err, color=C_TRAIN, lw=2.5, label='Train error')
ax2.plot(complexity, test_err,  color=C_OVER,  lw=2.5, label='Test error')
ax2.fill_between(complexity, train_err, test_err,
                 alpha=0.18, color=C_GAP, label='Overfit gap')

ax2.axvline(complexity[best], color=C_GOOD, lw=2, linestyle='--')

# Region labels
ax2.text(2.5, test_err[20]+0.15, 'High bias\n(underfit)',
         ha='center', color=C_AMBER, fontsize=10, fontweight='bold')
ax2.text(7.5, test_err[75]+0.15, 'High variance\n(overfit)',
         ha='center', color=C_OVER,  fontsize=10, fontweight='bold')

ax2.set_xlabel('Model complexity', color=SUBTEXT, fontsize=11)
ax2.set_ylabel('Error', color=SUBTEXT, fontsize=11)
ax2.set_title('Train vs Test Error\nThe gap is the diagnostic signal',
              fontsize=11, fontweight='bold', color=TEXT, pad=10)
ax2.legend(fontsize=10)

fig.text(0.5, 1.01, 'Bias-Variance Tradeoff',
         ha='center', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.975,
         'Expected error = Bias² + Variance + Irreducible noise only first two are fixable',
         ha='center', fontsize=10.5, color=SUBTEXT)

plt.savefig('figures_of/fig4_bias_variance.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show()
print('Figure 4 saved.')

Figure 4 saved.


## Figure 5 Diagnostic Flowchart

In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 10))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
ax.set_xlim(0, 13); ax.set_ylim(0, 13)
ax.axis('off')

NAVY_D = '1A1D2E'
styles = {
    'start':   ('#1a3a5c', '#4F9CF9'),
    'check':   ('#1E2140', '#FFB86C'),
    'good':    ('#0d2b1a', '#43D9AD'),
    'bad':     ('#2b0d0d', '#FF6B6B'),
    'fix':     ('#1f1030', '#BD93F9'),
    'action':  ('#1A1D2E', '#4F9CF9'),
}

def fbox(ax, cx, cy, w, h, lines, style='action'):
    fill, edge = styles[style]
    r = FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                        boxstyle='round,pad=0.15', lw=2.2,
                        facecolor=fill, edgecolor=edge, zorder=3)
    glow = FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                           boxstyle='round,pad=0.25', lw=4,
                           facecolor='none', edgecolor=edge, alpha=0.18, zorder=2)
    ax.add_patch(r); ax.add_patch(glow)
    text = '\n'.join(lines) if isinstance(lines, list) else lines
    col = '#E8EAF6' if style not in ('good','bad','fix') else edge
    ax.text(cx, cy, text, ha='center', va='center', fontsize=10,
            color=col, fontweight='bold', zorder=4, multialignment='center',
            linespacing=1.4)

def farrow(ax, x1, y1, x2, y2, label='', lc='#8892B0'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=lc, lw=2.0), zorder=2)
    if label:
        mx,my = (x1+x2)/2+0.25,(y1+y2)/2
        ax.text(mx,my,label,ha='left',va='center',fontsize=9.5,
                color=lc,fontweight='bold')

# Nodes
fbox(ax, 6.5, 12.3, 5.5, 0.85, ['START Model trained'], 'start')
fbox(ax, 6.5, 11.0, 6.5, 0.9,  ['Compare: Train accuracy  vs  Test accuracy'], 'check')

fbox(ax, 1.8, 9.3,  3.0, 0.85, ['Train ≈ Test', '→ Check for underfit'], 'good')
fbox(ax, 6.5, 9.3,  3.0, 0.85, ['Both low', '→ UNDERFIT'], 'bad')
fbox(ax, 11.2,9.3,  3.0, 0.85, ['Train >> Test', '→ OVERFIT'], 'bad')

fbox(ax, 6.5, 7.8,  4.5, 0.85, ['Diagnose with learning curve\n& validation curve'], 'action')

fbox(ax, 2.2, 6.3,  3.8, 0.85, ['Gap shrinks with data?\n→ Get more data'], 'fix')
fbox(ax, 10.8,6.3,  3.8, 0.85, ['Gap stays constant?\n→ Reduce complexity'], 'fix')

fbox(ax, 6.5, 4.9,  7.5, 1.1,
     ['Apply fixes in order:', '1. More data   2. Reduce complexity   3. Regularise   4. Early stopping   5. CV'], 'action')

fbox(ax, 6.5, 3.4,  6.0, 0.85, ['Re-evaluate: is gap < 2%?'], 'check')
fbox(ax, 2.5, 2.0,  3.5, 0.85, ['✓ DONE deploy model'], 'good')
fbox(ax, 10.5,2.0,  3.5, 0.85, ['↺ Iterate try next fix'], 'action')

# Tip box
tip = FancyBboxPatch((0.2, 0.2), 4.2, 1.4,
                      boxstyle='round,pad=0.2', lw=1.5,
                      facecolor='#0d2b1a', edgecolor='#43D9AD', alpha=0.9, zorder=3)
ax.add_patch(tip)
ax.text(2.3, 0.9, '💡 Key rules:\n• Always diagnose before fixing\n• Check learning curve first',
        ha='center', va='center', fontsize=9, color='#43D9AD', zorder=4, linespacing=1.5)

# Arrows
farrow(ax, 6.5, 11.87, 6.5, 11.45)
farrow(ax, 3.75, 11.0, 1.8, 9.75, 'GOOD')
farrow(ax, 6.5,  10.55, 6.5, 9.75, 'BOTH LOW')
farrow(ax, 9.25, 11.0, 11.2, 9.75, 'OVERFIT')
farrow(ax, 11.2, 8.87, 8.75, 8.25)
farrow(ax, 6.5,  8.87, 6.5,  8.25)
farrow(ax, 4.27, 7.8, 2.2, 6.75)
farrow(ax, 8.73, 7.8, 10.8, 6.75)
farrow(ax, 2.2,  5.87, 5.25, 5.45)
farrow(ax, 10.8, 5.87, 7.75, 5.45)
farrow(ax, 6.5,  4.44, 6.5,  3.85)
farrow(ax, 4.50, 3.4, 2.5, 2.42, 'YES')
farrow(ax, 8.50, 3.4, 10.5, 2.42, 'NO')

ax.set_title('Overfitting Diagnostic Flowchart\nA step-by-step guide to detecting and fixing overfitting',
             fontsize=12, fontweight='bold', color=TEXT, pad=10)

plt.savefig('figures_of/fig5_diagnostic_flowchart.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show()
print('Figure 5 saved.')

Figure 5 saved.


## Summary

| Signal | Meaning | Fix |
|--------|---------|-----|
| Train acc ≈ Test acc (both high) | Good fit | Deploy |
| Train acc >> Test acc | Overfitting high variance | More data, reduce complexity, regularise |
| Both low | Underfitting high bias | More capacity, more features, less regularisation |

**Diagnostic order:**
1. Plot learning curve does more data help?
2. Plot validation curve is the model too complex?
3. Apply the right fix from the table above
4. Re-evaluate iterate until gap < 2%